In [11]:
# Calculate average time spent on each process step

import pandas as pd

df = pd.read_csv("vanguard_master_cleaned.csv")
df.head()

,client_id,visitor_id,visit_id,process_step,date_time,Variation,clnt_tenure_yr,clnt_tenure_mnth,clnt_age,gendr,num_accts,bal,calls_6_mnth,logons_6_mnth,is_high_balance,is_multi_account
0,9988021,580560515_7732621733,781255054_21935453173_531117,step_3,2017-04-17 15:27:07,Test,5.0,64.0,79.0,U,2.0,189023.86,1.0,4.0,True,False
1,9988021,580560515_7732621733,781255054_21935453173_531117,step_2,2017-04-17 15:26:51,Test,5.0,64.0,79.0,U,2.0,189023.86,1.0,4.0,True,False
2,9988021,580560515_7732621733,781255054_21935453173_531117,step_3,2017-04-17 15:19:22,Test,5.0,64.0,79.0,U,2.0,189023.86,1.0,4.0,True,False
3,9988021,580560515_7732621733,781255054_21935453173_531117,step_2,2017-04-17 15:19:13,Test,5.0,64.0,79.0,U,2.0,189023.86,1.0,4.0,True,False
4,9988021,580560515_7732621733,781255054_21935453173_531117,step_3,2017-04-17 15:18:04,Test,5.0,64.0,79.0,U,2.0,189023.86,1.0,4.0,True,False


In [12]:
df["date_time"] = pd.to_datetime(df["date_time"])
df = df.sort_values(["visit_id", "date_time"]).copy()

In [13]:
df["next_date_time"] = df.groupby("visit_id")["date_time"].shift(-1)
df["next_step"] = df.groupby("visit_id")["process_step"].shift(-1)

df["time_spent_seconds"] = (df["next_date_time"] - df["date_time"]).dt.total_seconds()

In [14]:
time_df = df[
    (df["time_spent_seconds"].notna()) &
    (df["time_spent_seconds"] >= 0)
].copy()

In [15]:
time_by_step = time_df.groupby("process_step")["time_spent_seconds"].agg(
    count="count",
    mean_seconds="mean",
    median_seconds="median"
).round(2)

time_by_step

,count,mean_seconds,median_seconds
process_step,,,
confirm,6550,216.47,72.0
start,84961,61.60,15.0
step_1,61786,56.23,24.0
step_2,54080,89.97,62.0
step_3,44645,131.94,64.0


In [16]:
# Continue with abandonment

visit_summary = df.groupby("visit_id").agg(
    client_id=("client_id", "first"),
    is_high_balance=("is_high_balance", "first"),
    reached_confirm=("process_step", lambda x: (x == "confirm").any()),
    last_step=("process_step", "last")
).reset_index()

visit_summary.head()

,visit_id,client_id,is_high_balance,reached_confirm,last_step
0,100012776_37918976071_457913,3561384,False,True,confirm
1,100019538_17884295066_43909,7338123,False,True,confirm
2,100022086_87870757897_149620,2478628,False,True,confirm
3,100030127_47967100085_936361,105007,False,False,start
4,100037962_47432393712_705583,5623007,False,False,start


In [17]:
high_balance_abandoned = visit_summary[
    (visit_summary["is_high_balance"] == True) &
    (visit_summary["reached_confirm"] == False)
]

high_balance_abandoned.shape

(8325, 5)

In [18]:
high_balance_abandoned["last_step"].value_counts()

last_step
start     4972
step_1    1740
step_3    1002
step_2     611
Name: count, dtype: int64

In [19]:
round(high_balance_abandoned["last_step"].value_counts(normalize=True) * 100, 2)

last_step
start     59.72
step_1    20.90
step_3    12.04
step_2     7.34
Name: proportion, dtype: float64

## Phase 2 Findings

The average time-spent analysis shows that users spent the most time on the `confirm` step, with an average of **216.47 seconds**. Among the main process steps before confirmation, `step_3` had the highest average time spent at **131.94 seconds**, followed by `step_2` at **89.97 seconds**. This suggests that users may experience more hesitation or friction later in the process.

For high-balance clients, there were **8,325 abandoned visits**, meaning these visits did not reach the `confirm` step. The largest abandonment point was the `start` step, where **59.72%** of abandoned high-balance visits ended. Another **20.90%** abandoned at `step_1`.

Based on these results, in-context prompts are most needed early in the process, especially at `start` and `step_1`, because this is where most high-balance clients drop off. Additional support may also be useful at `step_3`, since users spend the longest time there before confirmation.